In [7]:
#imports
import os
import zipfile
from datetime import datetime
import requests
import pandas as pd
from google.transit import gtfs_realtime_pb2

In [8]:
#setup
BASE_DIR = "mta_project_data"
STATIC_DIR = f"{BASE_DIR}/gtfs_static"
REALTIME_DIR = f"{BASE_DIR}/gtfs_realtime_snapshots"

os.makedirs(STATIC_DIR, exist_ok=True)
os.makedirs(REALTIME_DIR, exist_ok=True)

# 1. Static GTFS Subway Dataset
STATIC_GTFS_URL = "https://rrgtfsfeeds.s3.amazonaws.com/gtfs_subway.zip"

# 2. Realtime GTFS feeds
REALTIME_FEEDS = {
    "1234567S": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs",
    "ACE": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-ace",
    "BDFM": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-bdfm",
    "G": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-g",
    "JZ": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-jz",
    "NQRW": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-nqrw",
    "L": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-l"
}

In [9]:
def download_static_gtfs():
    zip_path = f"{BASE_DIR}/gtfs_subway.zip"

    #print("Downloading static GTFS...")
    response = requests.get(STATIC_GTFS_URL)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(STATIC_DIR)

    #print("Static GTFS downloaded and extracted.")


def load_static_tables():
    print("\nStatic GTFS tables:")
    dflist = []
    for file in os.listdir(STATIC_DIR):
        if file.endswith(".txt"):
            path = f"{STATIC_DIR}/{file}"
            df = pd.read_csv(path)
            dflist.append(df)
            print(f"\n{file}")
            print(df.head())
            print("Columns:", list(df.columns))
    return dflist


def collect_realtime_snapshot():
    all_rows = []

    for feed_name, url in REALTIME_FEEDS.items():
        #print(f"Collecting realtime feed: {feed_name}")

        response = requests.get(url)
        response.raise_for_status()

        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.content)

        for entity in feed.entity:
            if entity.HasField("vehicle"):
                vehicle = entity.vehicle

                all_rows.append({
                    "feed_name": feed_name,
                    "entity_id": entity.id,
                    "trip_id": vehicle.trip.trip_id,
                    "route_id": vehicle.trip.route_id,
                    "direction_id": vehicle.trip.direction_id,
                    "start_time": vehicle.trip.start_time,
                    "start_date": vehicle.trip.start_date,
                    "schedule_relationship": vehicle.trip.schedule_relationship,
                    "current_stop_sequence": vehicle.current_stop_sequence,
                    "current_status": vehicle.current_status,
                    "stop_id": vehicle.stop_id,
                    "timestamp_unix": vehicle.timestamp,
                    "timestamp_readable": datetime.fromtimestamp(vehicle.timestamp) if vehicle.timestamp else None,
                    "vehicle_id": vehicle.vehicle.id,
                    "vehicle_label": vehicle.vehicle.label,
                    "latitude": vehicle.position.latitude,
                    "longitude": vehicle.position.longitude,
                    "bearing": vehicle.position.bearing,
                    "odometer": vehicle.position.odometer,
                    "speed": vehicle.position.speed,
                    "congestion_level": vehicle.congestion_level,
                    "occupancy_status": vehicle.occupancy_status
                })

    df = pd.DataFrame(all_rows)

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"{REALTIME_DIR}/mta_realtime_snapshot_{now}.csv"

    df.to_csv(output_path, index=False)

    #print("\nRealtime snapshot saved:")
    #print(output_path)

    #print("\nRealtime table preview:")
    #print(df.head())

    #print("\nColumns:")
    #print(list(df.columns))
    #print(df.isna().sum())

    return df


if __name__ == "__main__":
    download_static_gtfs()
    dflist = load_static_tables()
    realtime_df = collect_realtime_snapshot()


Static GTFS tables:

agency.txt
  agency_id                agency_name           agency_url   agency_timezone  \
0  MTA NYCT  MTA New York City Transit  http://www.mta.info  America/New_York   

  agency_lang  agency_phone  
0          en  718-330-1234  
Columns: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']

calendar.txt
  service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
0     Sunday       0        0          0         0       0         0       1   
1   Saturday       0        0          0         0       0         1       0   
2    Weekday       1        1          1         1       1         0       0   

   start_date  end_date  
0    20260301  20260516  
1    20260301  20260516  
2    20260301  20260516  
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

calendar_dates.txt
Empty DataFrame
Columns: [service_id, date, exception_t

In [ ]:
# TODO: add code for other stuff
print(dflist[0])

[  agency_id                agency_name           agency_url   agency_timezone  \
0  MTA NYCT  MTA New York City Transit  http://www.mta.info  America/New_York   

  agency_lang  agency_phone  
0          en  718-330-1234  ,   service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
0     Sunday       0        0          0         0       0         0       1   
1   Saturday       0        0          0         0       0         1       0   
2    Weekday       1        1          1         1       1         0       0   

   start_date  end_date  
0    20260301  20260516  
1    20260301  20260516  
2    20260301  20260516  , Empty DataFrame
Columns: [service_id, date, exception_type]
Index: [],    route_id agency_id route_short_name                 route_long_name  \
0         A  MTA NYCT                A                8 Avenue Express   
1         C  MTA NYCT                C                  8 Avenue Local   
2         E  MTA NYCT                E                  

In [ ]:
# =====================================================
# FEATURE ENGINEERING FOR TRAIN DELAY PREDICTION
# =====================================================
import numpy as np
import glob
from datetime import timedelta

# ---------- Load Static Data ----------
stop_times = pd.read_csv(f"{STATIC_DIR}/stop_times.txt")
stops = pd.read_csv(f"{STATIC_DIR}/stops.txt")
trips = pd.read_csv(f"{STATIC_DIR}/trips.txt")
calendar = pd.read_csv(f"{STATIC_DIR}/calendar.txt")

# ---------- Load Realtime Snapshot ----------
rt_files = sorted(glob.glob(f"{REALTIME_DIR}/mta_realtime_snapshot_*.csv"))
realtime = pd.read_csv(rt_files[-1]) if rt_files else pd.DataFrame()

# ---------- Build Today's Schedule ----------
schedule = stop_times.merge(
    trips[['trip_id', 'route_id', 'direction_id', 'service_id']], on='trip_id'
)

def time_to_sec(t):
    h, m, s = map(int, t.split(':'))
    return h * 3600 + m * 60 + s

schedule['arrival_sec'] = schedule['arrival_time'].apply(time_to_sec)
schedule['departure_sec'] = schedule['departure_time'].apply(time_to_sec)

date_str = str(realtime['start_date'].iloc[0]) if not realtime.empty else "20260512"
base_date = datetime.strptime(date_str, "%Y%m%d")
day_col = base_date.strftime("%A").lower()
active_svc = calendar[calendar[day_col] == 1]['service_id'].tolist()
sched = schedule[schedule['service_id'].isin(active_svc)].copy()

print(f"Date: {base_date.date()} ({base_date.strftime('%A')})")
print(f"Active services: {active_svc}")
print(f"Scheduled stop-times today: {len(sched):,}")
print(f"Realtime vehicle positions:  {len(realtime):,}\n")

# ==========================================================
# 1. STATION_ID — already exists as stop_id in both datasets
# ==========================================================
print("=" * 55)
print("1. STATION_ID — EXISTS")
print(f"   Static unique stops: {stops['stop_id'].nunique()}")
print(f"   Realtime unique stops: {realtime['stop_id'].nunique()}\n")

# ==========================================================
# 2 & 3. SCHEDULED ARRIVAL / DEPARTURE TIMESTAMPS
# ==========================================================
sched['sched_arrival_ts'] = sched['arrival_sec'].apply(
    lambda s: base_date + timedelta(seconds=int(s))
)
sched['sched_departure_ts'] = sched['departure_sec'].apply(
    lambda s: base_date + timedelta(seconds=int(s))
)

print("=" * 55)
print("2/3. SCHEDULED ARRIVAL & DEPARTURE TIMESTAMPS — COMPUTED")
print(sched[['trip_id', 'stop_id', 'sched_arrival_ts', 'sched_departure_ts', 'stop_sequence']].head())
print("     (Actual timestamps require TripUpdate — collected below)\n")

# ==========================================================
# 4. SCHEDULED HEADWAY
# ==========================================================
sched_s = sched.sort_values(['stop_id', 'route_id', 'direction_id', 'arrival_sec'])
sched_s['sched_headway_sec'] = sched_s.groupby(
    ['stop_id', 'route_id', 'direction_id']
)['arrival_sec'].diff()

hw = sched_s['sched_headway_sec'].dropna()
print("=" * 55)
print(f"4. SCHEDULED HEADWAY — COMPUTED ({len(hw):,} consecutive pairs)")
print(f"   Mean: {hw.mean():.0f}s | Median: {hw.median():.0f}s | "
      f"Range: [{hw.min():.0f}s, {hw.max():.0f}s]\n")

# ==========================================================
# 5. DWELL TIME (scheduled)
# ==========================================================
sched['sched_dwell_sec'] = sched['departure_sec'] - sched['arrival_sec']
print("=" * 55)
print(f"5. SCHEDULED DWELL TIME — {sched['sched_dwell_sec'].unique()}")
print("   MTA static schedule sets arrival == departure → dwell is always 0")
print("   (Actual dwell requires TripUpdate — collected below)\n")

# ==========================================================
# 6. TRAIN-TO-TRAIN SPACING (from realtime snapshot)
# ==========================================================
def parse_origin_sec(tid):
    try:
        t = tid.split('_')[0]
        return int(t[:2]) * 3600 + int(t[2:4]) * 60 + (int(t[4:6]) if len(t) >= 6 else 0)
    except Exception:
        return np.nan

rt = realtime.copy()
rt['origin_sec'] = rt['trip_id'].apply(parse_origin_sec)

rt_seq = rt.sort_values(['route_id', 'direction_id', 'current_stop_sequence'])
rt_seq['seq_spacing'] = rt_seq.groupby(
    ['route_id', 'direction_id']
)['current_stop_sequence'].diff()

rt_t = rt.sort_values(['route_id', 'direction_id', 'origin_sec'])
rt_t['time_spacing_sec'] = rt_t.groupby(
    ['route_id', 'direction_id']
)['origin_sec'].diff()

ts_vals = rt_t['time_spacing_sec'].dropna()
print("=" * 55)
print("6. TRAIN-TO-TRAIN SPACING — COMPUTED (from snapshot)")
print(f"   Stop-sequence spacing mean: {rt_seq['seq_spacing'].mean():.1f} stops")
print(f"   Time spacing — Mean: {ts_vals.mean():.0f}s | Median: {ts_vals.median():.0f}s\n")

# ==========================================================
# 7 & 8. COLLECT TRIP UPDATES for actual arrival/departure,
#         delay propagation, and sequence modeling
# ==========================================================
print("=" * 55)
print("COLLECTING TRIP UPDATES FROM GTFS-RT FEEDS...")
print("(Your current code only collects VehiclePosition;")
print(" TripUpdate entities have per-stop arrival/departure predictions)")
print("=" * 55)

def collect_trip_updates():
    rows = []
    for feed_name, url in REALTIME_FEEDS.items():
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            feed = gtfs_realtime_pb2.FeedMessage()
            feed.ParseFromString(resp.content)
            for ent in feed.entity:
                if ent.HasField("trip_update"):
                    tu = ent.trip_update
                    for stu in tu.stop_time_update:
                        rows.append({
                            "feed": feed_name,
                            "trip_id": tu.trip.trip_id,
                            "route_id": tu.trip.route_id,
                            "direction_id": tu.trip.direction_id,
                            "start_date": tu.trip.start_date,
                            "stop_id": stu.stop_id,
                            "stop_sequence": stu.stop_sequence,
                            "arrival_time": stu.arrival.time if stu.HasField("arrival") else None,
                            "arrival_delay": stu.arrival.delay if stu.HasField("arrival") else None,
                            "departure_time": stu.departure.time if stu.HasField("departure") else None,
                            "departure_delay": stu.departure.delay if stu.HasField("departure") else None,
                        })
        except Exception as e:
            print(f"  {feed_name}: {e}")
    return pd.DataFrame(rows)

tu = collect_trip_updates()
print(f"\nCollected {len(tu):,} stop-time update records")

if not tu.empty:
    tu.to_csv(
        f"{REALTIME_DIR}/mta_trip_updates_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv",
        index=False,
    )

    # --- 2/3 revisited: Actual arrival & departure timestamps ---
    tu['actual_arrival_ts'] = pd.to_datetime(tu['arrival_time'], unit='s', errors='coerce')
    tu['actual_departure_ts'] = pd.to_datetime(tu['departure_time'], unit='s', errors='coerce')

    print("\n--- Actual Arrival/Departure Timestamps (TripUpdate) ---")
    print(tu[['trip_id', 'stop_id', 'actual_arrival_ts', 'actual_departure_ts']].dropna().head())

    # --- 5 revisited: Actual dwell time ---
    tu['actual_dwell_sec'] = (
        tu['departure_time'].astype(float) - tu['arrival_time'].astype(float)
    )
    dw = tu['actual_dwell_sec'].dropna()
    dw = dw[dw >= 0]
    print(f"\n--- Actual Dwell Time ---")
    if len(dw) > 0:
        print(f"   Mean: {dw.mean():.1f}s | Median: {dw.median():.1f}s | n={len(dw):,}")
    else:
        print("   Not computable (arrival or departure times missing in this feed)")

    # --- 4 revisited: Actual headway ---
    tu_h = tu.dropna(subset=['arrival_time']).sort_values(
        ['stop_id', 'route_id', 'direction_id', 'arrival_time']
    )
    tu_h['actual_headway_sec'] = tu_h.groupby(
        ['stop_id', 'route_id', 'direction_id']
    )['arrival_time'].diff()
    ah = tu_h['actual_headway_sec'].dropna()
    print(f"\n--- Actual Headway ---")
    print(f"   Mean: {ah.mean():.0f}s | Median: {ah.median():.0f}s | n={len(ah):,}")

    # --- 7. Per-Station Delay Propagation ---
    print(f"\n--- Per-Station Delay Propagation ---")
    has_delay = tu['arrival_delay'].notna().any() and (tu['arrival_delay'] != 0).any()

    if has_delay:
        stn_delay = tu.groupby('stop_id')['arrival_delay'].agg(['mean', 'std', 'count'])
        stn_delay.columns = ['mean_delay_s', 'std_delay_s', 'n_trains']
        stn_delay = stn_delay.sort_values('mean_delay_s', ascending=False)
        top = stn_delay.head(10).merge(
            stops[['stop_id', 'stop_name']], left_index=True, right_on='stop_id', how='left'
        )
        print(f"   Stations with delay data: {len(stn_delay)}")
        print("   Top 10 most delayed:")
        print(top[['stop_id', 'stop_name', 'mean_delay_s', 'n_trains']].to_string(index=False))

        trip_del = tu.dropna(subset=['arrival_delay']).sort_values(['trip_id', 'stop_sequence'])
        trip_del['delay_change'] = trip_del.groupby('trip_id')['arrival_delay'].diff()
        dc = trip_del['delay_change'].dropna()
        print(f"\n   Delay propagation between consecutive stops:")
        print(f"   Mean change: {dc.mean():.1f}s | Std: {dc.std():.1f}s")
    else:
        print("   arrival_delay field not populated; computing delay from schedule match...")
        def time_to_sec_from_digits(d):
            return int(d[:2]) * 3600 + int(d[2:4]) * 60 + (int(d[4:6]) if len(d) >= 6 else 0)

        tu['origin_sec'] = tu['trip_id'].apply(parse_origin_sec)
        tu['shape'] = tu['trip_id'].apply(
            lambda x: x.split('_', 1)[1] if '_' in x else None
        )
        sched['shape'] = sched['trip_id'].apply(
            lambda x: x.rsplit('_', 1)[-1] if '_' in x else None
        )
        sched['origin_sec_static'] = sched['trip_id'].apply(lambda x: (
            time_to_sec_from_digits(x.rsplit('_', 2)[-2])
            if len(x.rsplit('_', 2)) >= 3 and x.rsplit('_', 2)[-2].isdigit()
            else np.nan
        ))

        delays = []
        for (origin, shape), grp in tu.groupby(['origin_sec', 'shape']):
            match = sched[
                (sched['origin_sec_static'] == origin) & (sched['shape'] == shape)
            ]
            if match.empty:
                continue
            merged = grp.merge(
                match[['stop_id', 'stop_sequence', 'arrival_sec']],
                on=['stop_id', 'stop_sequence'],
                how='inner',
            )
            if merged.empty:
                continue
            merged['sched_unix'] = base_date.timestamp() + merged['arrival_sec']
            merged['delay_sec'] = merged['arrival_time'].astype(float) - merged['sched_unix']
            delays.append(merged[['trip_id', 'stop_id', 'stop_sequence', 'delay_sec']])

        if delays:
            delay_df = pd.concat(delays)
            stn_delay = delay_df.groupby('stop_id')['delay_sec'].agg(['mean', 'std', 'count'])
            stn_delay.columns = ['mean_delay_s', 'std_delay_s', 'n_trains']
            stn_delay = stn_delay.sort_values('mean_delay_s', ascending=False)
            top = stn_delay.head(10).merge(
                stops[['stop_id', 'stop_name']], left_index=True, right_on='stop_id', how='left'
            )
            print(f"   Matched {len(delay_df):,} stop records to schedule")
            print("   Top 10 most delayed stations:")
            print(top[['stop_id', 'stop_name', 'mean_delay_s', 'n_trains']].to_string(index=False))

            delay_df_sorted = delay_df.sort_values(['trip_id', 'stop_sequence'])
            delay_df_sorted['delay_change'] = delay_df_sorted.groupby('trip_id')['delay_sec'].diff()
            dc = delay_df_sorted['delay_change'].dropna()
            print(f"\n   Delay propagation between consecutive stops:")
            print(f"   Mean change: {dc.mean():.1f}s | Std: {dc.std():.1f}s")
        else:
            print("   Could not match realtime trips to schedule")

    # --- 8. Sequence Modeling Features ---
    print(f"\n--- Sequence Modeling Features ---")
    seq = tu.sort_values(['trip_id', 'stop_sequence'])
    seq_agg = seq.groupby('trip_id').agg(
        n_stops=('stop_id', 'count'),
        stops=('stop_id', list),
        delays=('arrival_delay', list),
        arrivals=('arrival_time', list),
    ).reset_index()
    print(f"   Trip sequences built: {len(seq_agg):,}")
    print(f"   Mean stops per sequence: {seq_agg['n_stops'].mean():.1f}")
    print(f"   Example: {seq_agg.iloc[0]['trip_id']} → {seq_agg.iloc[0]['stops'][:6]}...")

else:
    print("No TripUpdate data collected — API may be unavailable.")
    print("TripUpdate is REQUIRED for: actual timestamps, actual dwell, delay propagation, sequences")

# ==========================================================
# FINAL SUMMARY
# ==========================================================
has_tu = not tu.empty if 'tu' in dir() else False
has_delay_col = has_tu and tu['arrival_delay'].notna().any() and (tu['arrival_delay'] != 0).any()

print(f"\n{'=' * 55}")
print("FEATURE AVAILABILITY SUMMARY")
print("=" * 55)
features = [
    ("station_id",                "EXISTS (stop_id in both datasets)"),
    ("sched_arrival_timestamp",   "COMPUTED from stop_times + date"),
    ("sched_departure_timestamp", "COMPUTED from stop_times + date"),
    ("actual_arrival_timestamp",  "COMPUTED (TripUpdate)" if has_tu else "NEEDS TripUpdate API"),
    ("actual_departure_timestamp","COMPUTED (TripUpdate)" if has_tu else "NEEDS TripUpdate API"),
    ("scheduled_headway",         "COMPUTED from stop_times"),
    ("actual_headway",            "COMPUTED (TripUpdate)" if has_tu else "NEEDS TripUpdate API"),
    ("scheduled_dwell_time",      "ALWAYS 0 — MTA sets arrival == departure"),
    ("actual_dwell_time",         "COMPUTED (TripUpdate)" if has_tu else "NEEDS TripUpdate API"),
    ("train_to_train_spacing",    "COMPUTED from snapshot"),
    ("per_station_delay",         "COMPUTED" if (has_tu) else "NEEDS TripUpdate"),
    ("delay_propagation",         "COMPUTED" if (has_tu) else "NEEDS TripUpdate"),
    ("sequence_features",         "COMPUTED (TripUpdate)" if has_tu else "NEEDS TripUpdate"),
]
for name, status in features:
    print(f"  {name:30s} → {status}")